# Color-sampled Pixasonics video sonification

This notebook uses the image-frame workflow from the Pixasonics project, but plays your **recorded MP3 samples** instead of a Pixasonics synth. Pixasonics' standard Mapper controls synth parameters rather than selecting audio files, so the color analysis and sample mixing below are done with OpenCV, NumPy, and FFmpeg. Each scanline is analyzed in HSV color space; red, orange, yellow, and blue pixel proportions control the mix of `red.mp3`, `orange.mp3`, `sarı.mp3`, and `blue.mp3`. White/whitish regions trigger the kick sample once when first reached by the scanning line. The output is `sonification.wav` plus a video with the matching moving line and audio.

Expected files next to this notebook: `frames_0/`, `red.mp3`, `orange.mp3`, `sarı.mp3`, `blue.mp3`, and `ayudharamusic-kick-183936.mp3`. Edit the paths in the configuration cell if yours differ. Bright white, off-white, and light gray connected regions trigger one quieter kick when the scanline first reaches them. Requires `opencv-python`, `numpy`, and FFmpeg (`brew install ffmpeg` on macOS).

In [1]:
from pathlib import Path
import re
import shutil
import subprocess
import wave

import cv2
import numpy as np

# Edit these paths if your files are in different folders.
FRAME_DIR = Path("./frames_0")
RED_SAMPLE = Path("./red.mp3")
ORANGE_SAMPLE = Path("./orange.mp3")
YELLOW_SAMPLE = Path("./sarı.mp3")
BLUE_SAMPLE = Path("./blue.mp3")
KICK_SAMPLE = Path("./ayudharamusic-kick-183936.mp3")

AUDIO_PATH = Path("sonification.wav")
TEMP_VIDEO_PATH = Path("frames_video.mp4")
OUTPUT_PATH = Path("color_sampled_pixasonics.mp4")

VIDEO_FPS = 30
SECONDS_PER_IMAGE = 3.0
IMAGE_SIZE = 500
SAMPLE_RATE = 44100
LINE_THICKNESS = 2

FFMPEG = shutil.which("ffmpeg")
if FFMPEG is None:
    raise FileNotFoundError("FFmpeg not found. On macOS, install it with `brew install ffmpeg`, then restart the notebook kernel.")

for sample_path in (RED_SAMPLE, ORANGE_SAMPLE, YELLOW_SAMPLE, BLUE_SAMPLE, KICK_SAMPLE):
    if not sample_path.is_file():
        raise FileNotFoundError(f"Audio sample not found: {sample_path.resolve()}")
if not FRAME_DIR.is_dir():
    raise FileNotFoundError(f"Frame folder not found: {FRAME_DIR.resolve()}")

def natural_sort_key(path):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", path.name)]

frame_paths = sorted(
    [p for p in FRAME_DIR.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}],
    key=natural_sort_key,
)
if not frame_paths:
    raise FileNotFoundError(f"No PNG/JPG/TIFF frames found in {FRAME_DIR.resolve()}")

frames_per_image = round(SECONDS_PER_IMAGE * VIDEO_FPS)
if frames_per_image < 1:
    raise ValueError("SECONDS_PER_IMAGE * VIDEO_FPS must be at least 1.")
segment_duration = frames_per_image / VIDEO_FPS
total_frames = len(frame_paths) * frames_per_image
total_duration = total_frames / VIDEO_FPS
print(f"Found {len(frame_paths)} input frames; output duration is {total_duration:.1f} seconds.")

Found 10 input frames; output duration is 30.0 seconds.


## Decode the MP3 samples and detect colors

FFmpeg decodes each MP3 to 44.1 kHz stereo float audio. Each sample is RMS-level matched to a shared dBFS target (with a peak ceiling) before mixing, so one louder recording is less likely to dominate. This is amplitude/RMS matching rather than broadcast LUFS normalization. Color classification uses overlapping hue memberships, and mix weights ease toward each new scanline over a configurable time constant.

In [2]:
def decode_mp3(path):
    result = subprocess.run(
        [FFMPEG, "-v", "error", "-i", str(path), "-f", "f32le", "-acodec", "pcm_f32le", "-ac", "2", "-ar", str(SAMPLE_RATE), "pipe:1"],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Could not decode {path}: {result.stderr.decode(errors='replace')}")
    audio = np.frombuffer(result.stdout, dtype="<f4").copy()
    if audio.size == 0 or audio.size % 2:
        raise ValueError(f"Decoded audio is empty or malformed: {path}")
    audio = audio.reshape(-1, 2)
    if not np.isfinite(audio).all():
        raise ValueError(f"Audio contains invalid values: {path}")
    return audio

SAMPLE_TARGET_DBFS = -24.0
SAMPLE_PEAK_CEILING_DBFS = -1.0
SAMPLE_EXTRA_GAIN_DB = {"red": 4.8, "blue": -6.0}
BLUE_YELLOW_TRANSITION_PATH = Path("blue_to_yellow_transition.mp3")
BLUE_YELLOW_CROSSFADE_SECONDS = 4.0

def match_sample_level(audio, target_dbfs=SAMPLE_TARGET_DBFS, peak_ceiling_dbfs=SAMPLE_PEAK_CEILING_DBFS):
    """Match each sample's RMS level while preventing excessive peaks."""
    rms = float(np.sqrt(np.mean(np.square(audio, dtype=np.float64))))
    peak = float(np.max(np.abs(audio)))
    if rms <= 1e-8 or peak <= 1e-8:
        raise ValueError("An audio sample is silent and cannot be level-matched.")

    target_rms = 10.0 ** (target_dbfs / 20.0)
    peak_ceiling = 10.0 ** (peak_ceiling_dbfs / 20.0)
    gain = min(target_rms / rms, peak_ceiling / peak)
    return (audio * gain).astype(np.float32), gain

def make_loopable(audio, crossfade_ms=40):
    fade = min(int(SAMPLE_RATE * crossfade_ms / 1000), len(audio) // 4)
    if fade < 2:
        return audio
    fade_in = np.linspace(0.0, 1.0, fade, endpoint=False, dtype=np.float32)[:, None]
    blended = audio[-fade:] * (1.0 - fade_in) + audio[:fade] * fade_in
    return np.concatenate((audio[fade:-fade], blended), axis=0)

def trim_leading_silence(audio, threshold_db_below_peak=-18.0, window_ms=10, preroll_ms=20):
    """Remove encoded leading silence while preserving a short pre-onset buffer."""
    peak = float(np.max(np.abs(audio)))
    if peak <= 1e-8:
        raise ValueError("An audio sample is silent and cannot be trimmed.")
    window_size = max(1, round(SAMPLE_RATE * window_ms / 1000.0))
    threshold = peak * 10.0 ** (threshold_db_below_peak / 20.0)
    window_rms = np.array(
        [
            np.sqrt(np.mean(np.square(audio[start:start + window_size], dtype=np.float64)))
            for start in range(0, len(audio) - window_size + 1, window_size)
        ]
    )
    active_windows = np.flatnonzero(window_rms >= threshold)
    if not active_windows.size:
        return audio
    onset = max(0, int(active_windows[0]) * window_size - round(SAMPLE_RATE * preroll_ms / 1000.0))
    return audio[onset:]

sample_paths = {
    "red": RED_SAMPLE,
    "orange": ORANGE_SAMPLE,
    "yellow": YELLOW_SAMPLE,
    "blue": BLUE_SAMPLE,
}
samples = {}
for color_name, sample_path in sample_paths.items():
    decoded = trim_leading_silence(decode_mp3(sample_path))
    samples[color_name], gain = match_sample_level(decoded)
    extra_gain = 10.0 ** (SAMPLE_EXTRA_GAIN_DB.get(color_name, 0.0) / 20.0)
    samples[color_name] *= extra_gain
    gain *= extra_gain
    print(
        f"{color_name}: {len(decoded) / SAMPLE_RATE:.2f}s, "
        f"level adjustment {20 * np.log10(gain):+.1f} dB"
    )
    samples[color_name] = make_loopable(samples[color_name])

KICK_EXTRA_GAIN_DB = -6.0
KICK_MIN_INTERVAL_MS = 600
kick_audio, kick_gain = match_sample_level(decode_mp3(KICK_SAMPLE))
kick_audio *= 10.0 ** (KICK_EXTRA_GAIN_DB / 20.0)
kick_gain *= 10.0 ** (KICK_EXTRA_GAIN_DB / 20.0)
print(f"kick: level adjustment {20 * np.log10(kick_gain):+.1f} dB")
WHITE_MIN_VALUE = 185
WHITE_MAX_SATURATION = 70
WHITE_MIN_AREA_PIXELS = 30

blue_audio = samples["blue"]
yellow_audio = samples["yellow"]
crossfade_samples = min(
    round(BLUE_YELLOW_CROSSFADE_SECONDS * SAMPLE_RATE),
    len(blue_audio),
    len(yellow_audio),
)
if crossfade_samples < 2:
    raise ValueError("The blue-yellow crossfade must contain at least two audio samples.")
fade_positions = np.linspace(0.0, np.pi / 2.0, crossfade_samples, dtype=np.float32)
blue_fade = np.cos(fade_positions)[:, None]
yellow_fade = np.sin(fade_positions)[:, None]
transition_audio = np.concatenate(
    (
        blue_audio[:-crossfade_samples],
        blue_audio[-crossfade_samples:] * blue_fade + yellow_audio[:crossfade_samples] * yellow_fade,
        yellow_audio[crossfade_samples:],
    ),
    axis=0,
)
transition_peak = float(np.max(np.abs(transition_audio)))
transition_ceiling = 10.0 ** (SAMPLE_PEAK_CEILING_DBFS / 20.0)
if transition_peak > transition_ceiling:
    transition_audio *= transition_ceiling / transition_peak
encode_result = subprocess.run(
    [
        FFMPEG, "-y", "-v", "error",
        "-f", "f32le", "-ar", str(SAMPLE_RATE), "-ac", "2", "-i", "pipe:0",
        "-codec:a", "libmp3lame", "-q:a", "2", str(BLUE_YELLOW_TRANSITION_PATH),
    ],
    input=transition_audio.astype("<f4").tobytes(),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    check=False,
)
if encode_result.returncode != 0:
    raise RuntimeError(
        f"Could not create {BLUE_YELLOW_TRANSITION_PATH}: "
        f"{encode_result.stderr.decode(errors='replace')}"
    )
print(
    f"Saved blue-to-yellow transition: {BLUE_YELLOW_TRANSITION_PATH.resolve()} "
    f"({len(transition_audio) / SAMPLE_RATE:.1f}s total, "
    f"{crossfade_samples / SAMPLE_RATE:.1f}s crossfade)"
)

COLOR_NAMES = ("red", "orange", "yellow", "blue")
# OpenCV hue is 0..179. Overlapping radii let color boundaries crossfade.
HUE_CENTERS = np.array([0, 16, 31, 110], dtype=np.float32)
HUE_RADII = np.array([20, 22, 24, 38], dtype=np.float32)
COLOR_MEMBERSHIP_SHARPNESS = 3.0
COLOR_SMOOTHING_MS = 180

def color_mix_weights(frame_bgr, y):
    """Return fuzzy red/orange/yellow/blue weights for one scanline."""
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    row = hsv[y].astype(np.float32)
    hue = row[:, 0]
    saturation = row[:, 1] / 255.0
    value = row[:, 2] / 255.0

    hue_delta = np.abs(hue[:, None] - HUE_CENTERS[None, :])
    hue_distance = np.minimum(hue_delta, 180.0 - hue_delta)
    memberships = np.clip(1.0 - hue_distance / HUE_RADII[None, :], 0.0, 1.0)
    memberships = memberships ** COLOR_MEMBERSHIP_SHARPNESS

    # Ignore nearly gray/dark pixels, but fade their influence in smoothly.
    confidence = np.clip((saturation - 0.08) / 0.35, 0.0, 1.0)
    confidence *= np.clip((value - 0.04) / 0.25, 0.0, 1.0)
    membership_sum = memberships.sum(axis=1, keepdims=True)
    per_pixel = np.divide(
        memberships,
        membership_sum,
        out=np.zeros_like(memberships),
        where=membership_sum > 0,
    )
    color_amounts = (per_pixel * confidence[:, None]).sum(axis=0)
    total = float(color_amounts.sum())
    if total <= 1e-8:
        return np.zeros(len(COLOR_NAMES), dtype=np.float32)
    return (color_amounts / total).astype(np.float32)

def looped_audio_chunk(sample, start_sample, count):
    indices = (np.arange(count, dtype=np.int64) + start_sample) % len(sample)
    return sample[indices]

def sample_mix(target_weights, current_weights, start_sample, count):
    """Mix samples while smoothly easing color gains toward the target."""
    tau_samples = max(1.0, SAMPLE_RATE * COLOR_SMOOTHING_MS / 1000.0)
    progress = 1.0 - np.exp(-np.arange(1, count + 1, dtype=np.float32) / tau_samples)
    weight_curve = current_weights[None, :] + progress[:, None] * (target_weights - current_weights)[None, :]
    result = np.zeros((count, 2), dtype=np.float32)
    for color_index, name in enumerate(COLOR_NAMES):
        if np.any(weight_curve[:, color_index] > 1e-6):
            result += looped_audio_chunk(samples[name], start_sample, count) * weight_curve[:, color_index, None]
    return result, weight_curve[-1].copy()

red: 5.65s, level adjustment -9.0 dB
orange: 33.22s, level adjustment -8.3 dB
yellow: 78.52s, level adjustment +0.9 dB
blue: 41.49s, level adjustment -18.9 dB
kick: level adjustment -15.0 dB
Saved blue-to-yellow transition: /Users/ozgeyavuz/Documents/SpaceAppsChallengeJukeBox/blue_to_yellow_transition.mp3 (115.9s total, 4.0s crossfade)


## Render the moving scan, sample-mixed audio, and final video

Each output video frame uses the next source image (held for `SECONDS_PER_IMAGE`) and a scanline that moves over the whole sequence. Fuzzy color memberships blend adjacent samples, and the gain envelope smooths changes over `COLOR_SMOOTHING_MS`. The saved source images are left unchanged apart from resizing; the red scanline is an overlay.

In [3]:
def scan_y(progress):
    progress = float(np.clip(progress, 0.0, 1.0))
    return round((IMAGE_SIZE - 1) * (1.0 - abs(2.0 * progress - 1.0)))

writer = cv2.VideoWriter(
    str(TEMP_VIDEO_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    VIDEO_FPS,
    (IMAGE_SIZE, IMAGE_SIZE),
)
if not writer.isOpened():
    raise RuntimeError(f"Could not create temporary video: {TEMP_VIDEO_PATH.resolve()}")

audio_blocks = []
kick_events = []
last_kick_sample = -round(SAMPLE_RATE * KICK_MIN_INTERVAL_MS / 1000.0)
current_mix_weights = np.zeros(len(COLOR_NAMES), dtype=np.float32)
samples_per_video_frame = round(SAMPLE_RATE / VIDEO_FPS)
audio_cursor = 0
try:
    for image_index, frame_path in enumerate(frame_paths):
        frame_bgr = cv2.imread(str(frame_path), cv2.IMREAD_COLOR)
        if frame_bgr is None:
            raise ValueError(f"Could not read image frame: {frame_path}")
        frame_bgr = cv2.resize(frame_bgr, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        frame_hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
        white_mask = cv2.inRange(
            frame_hsv,
            np.array([0, 0, WHITE_MIN_VALUE], dtype=np.uint8),
            np.array([179, WHITE_MAX_SATURATION, 255], dtype=np.uint8),
        )
        component_count, white_labels, component_stats, _ = cv2.connectedComponentsWithStats(
            white_mask,
            connectivity=8,
        )
        valid_white_components = {
            label
            for label in range(1, component_count)
            if component_stats[label, cv2.CC_STAT_AREA] >= WHITE_MIN_AREA_PIXELS
        }
        triggered_components = set()
        previous_y = None

        for subframe in range(frames_per_image):
            progress = (image_index + subframe / frames_per_image) / len(frame_paths)
            y = scan_y(progress)
            weights = color_mix_weights(frame_bgr, y)
            first_swept_y = y if previous_y is None else min(previous_y, y)
            last_swept_y = y if previous_y is None else max(previous_y, y)
            crossed_labels = np.unique(white_labels[first_swept_y:last_swept_y + 1, :])
            new_components = valid_white_components.intersection(crossed_labels.tolist()) - triggered_components
            min_kick_interval_samples = round(SAMPLE_RATE * KICK_MIN_INTERVAL_MS / 1000.0)
            if new_components and audio_cursor - last_kick_sample >= min_kick_interval_samples:
                kick_events.append(audio_cursor)
                last_kick_sample = audio_cursor
            triggered_components.update(new_components)
            previous_y = y

            block, current_mix_weights = sample_mix(
                weights,
                weights if audio_cursor == 0 else current_mix_weights,
                audio_cursor,
                samples_per_video_frame,
            )
            audio_blocks.append(block)
            audio_cursor += samples_per_video_frame

            display_frame = frame_bgr.copy()
            cv2.line(display_frame, (0, y), (IMAGE_SIZE - 1, y), (0, 0, 255), LINE_THICKNESS)
            writer.write(display_frame)

        print(f"Processed {image_index + 1}/{len(frame_paths)}: {frame_path.name}")
finally:
    writer.release()

if not audio_blocks:
    raise RuntimeError("No frames were processed.")

audio = np.concatenate(audio_blocks, axis=0)
print(f"Triggered {len(kick_events)} kick(s) from white/whitish regions.")
for kick_start in kick_events:
    kick_end = min(kick_start + len(kick_audio), len(audio))
    audio[kick_start:kick_end] += kick_audio[:kick_end - kick_start]
peak = float(np.max(np.abs(audio)))
if peak > 0:
    audio = audio / peak * 0.95
pcm = (np.clip(audio, -1.0, 1.0) * 32767).astype("<i2")
with wave.open(str(AUDIO_PATH), "wb") as wav_file:
    wav_file.setnchannels(2)
    wav_file.setsampwidth(2)
    wav_file.setframerate(SAMPLE_RATE)
    wav_file.writeframes(pcm.tobytes())

subprocess.run(
    [
        FFMPEG, "-y", "-v", "error",
        "-i", str(TEMP_VIDEO_PATH),
        "-i", str(AUDIO_PATH),
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "192k", "-shortest",
        str(OUTPUT_PATH),
    ],
    check=True,
)

print(f"Saved video: {OUTPUT_PATH.resolve()}")
print(f"Saved audio: {AUDIO_PATH.resolve()}")
print(f"Duration: {total_duration:.1f} seconds at {VIDEO_FPS} fps")

Processed 1/10: 00000001.png
Processed 2/10: 00000002.png
Processed 3/10: 00000003.png
Processed 4/10: 00000004.png
Processed 5/10: 00000005.png
Processed 6/10: 00000006.png
Processed 7/10: 00000007.png
Processed 8/10: 00000008.png
Processed 9/10: 00000009.png
Processed 10/10: 00000010.png
Triggered 7 kick(s) from white/whitish regions.
Saved video: /Users/ozgeyavuz/Documents/SpaceAppsChallengeJukeBox/color_sampled_pixasonics.mp4
Saved audio: /Users/ozgeyavuz/Documents/SpaceAppsChallengeJukeBox/sonification.wav
Duration: 30.0 seconds at 30 fps
